## **Importacion de Librerias necesarias**

In [1]:
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os
import random

## **Parametros de la Curva**

In [2]:
p = 97
a = 2
b = 3
G = (3, 6)

print("Curva: y^2 = x^3 + {}x + {} mod {}".format(a,b,p))
print("Generador G =", G)

Curva: y^2 = x^3 + 2x + 3 mod 97
Generador G = (3, 6)


## **Funciones**

In [3]:
def inv_mod(k, p):
    return pow(k % p, -1, p)

def inv_mod(k, p):
    return pow(k % p, -1, p)

def suma(P, Q):
    if P is None:
        return Q
    if Q is None:
        return P
    
    x1, y1 = P
    x2, y2 = Q

    if x1 == x2 and (y1 + y2) % p == 0:
        return None

    if P != Q:
        m = ((y2 - y1) * inv_mod(x2 - x1, p)) % p
    else:
        if y1 == 0:
            return None
        m = ((3*x1*x1 + a) * inv_mod(2*y1, p)) % p

    x3 = (m*m - x1 - x2) % p
    y3 = (m*(x1 - x3) - y1) % p
    return (x3, y3)

def mult(k, P):
    R = None
    addend = P
    
    while k > 0:
        if k & 1:
            R = suma(R, addend)
        addend = suma(addend, addend)
        k >>= 1
    return R

## **Generacion de claves validas**

In [4]:
def generar_claves_validas():
    while True:
        priv = random.randint(1, 20)
        pub = mult(priv, G)
        if pub is not None:
            return priv, pub
        
alice_priv, alice_pub = generar_claves_validas()
bob_priv, bob_pub = generar_claves_validas()

print("Priv Alice:", alice_priv)
print("Pub Alice:", alice_pub)
print()
print("Priv Bob:", bob_priv)
print("Pub Bob:", bob_pub)

Priv Alice: 6
Pub Alice: (3, 6)

Priv Bob: 11
Pub Bob: (3, 6)


## **Calculo de Clave Compartida**

In [5]:
while True:
    alice_shared = mult(alice_priv, bob_pub)
    bob_shared = mult(bob_priv, alice_pub)
    
    if alice_shared is not None and alice_shared == bob_shared:
        break
    else:
        alice_priv, alice_pub = generar_claves_validas()
        bob_priv, bob_pub = generar_claves_validas()

print("Shared Alice:", alice_shared)
print("Shared Bob:", bob_shared)
print("¿Coinciden?:", alice_shared == bob_shared)

Shared Alice: (3, 6)
Shared Bob: (3, 6)
¿Coinciden?: True
